## **Naive Bayes - Gauss**
Para datos continuos (ej. medidas biologicas)

In [ ]:
# Importamos todas las librerias necesarias para los 3 modelos Naive Bayes
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.preprocessing import Binarizer, StandardScaler
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.datasets import load_breast_cancer, fetch_20newsgroups, load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

In [ ]:
#Cargamos el dataset breast_cancer
#569 muestras con 30 caracteristicas numericas de tumores
datos_g = load_breast_cancer()
X_g = datos_g.data
y_g = datos_g.target  # 0 = maligno, 1 = benigno

#Verificamos si hay valores nulos en el dataset
df_g = pd.DataFrame(X_g, columns=datos_g.feature_names)
print("Valores nulos por columna:")
print(df_g.isnull().sum().sum(), "valores nulos en total")

#Escalamos las caracteristicas con StandardScaler
#Esto es importante para que todas las variables tengan la misma escala
#y el modelo no se vea afectado por diferencias de magnitud entre ellas
scaler_g = StandardScaler()
X_g_scaled = scaler_g.fit_transform(X_g)

print(f"\nForma del dataset: {X_g_scaled.shape}")
print(f"Distribucion de clases: Maligno={sum(y_g==0)}, Benigno={sum(y_g==1)}")

# Dividimos en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X_g_scaled, y_g, test_size=0.2, random_state=19
)

In [ ]:
#Creamos y entrenamos el modelo Gaussiano
#GaussianNB asume que las caracteristicas siguen una distribucion normal
#Es ideal para datos continuos como las medidas biologicas del dataset breast_cancer
modelo = GaussianNB()
modelo.fit(X_train, y_train)

In [ ]:
#Generamos las predicciones sobre el conjunto de prueba
y_pred = modelo.predict(X_test)

#predict_proba devuelve la probabilidad de cada clase
#Tomamos la columna [:, 1] que corresponde a la clase positiva (benigno)
#Necesario para calcular AUC-ROC
y_prob = modelo.predict_proba(X_test)[:, 1]

In [ ]:
#metricas principales del modelo Gaussiano
acc_g = accuracy_score(y_test, y_pred)
rec_g = recall_score(y_test, y_pred)
f1_g  = f1_score(y_test, y_pred)
auc_g = roc_auc_score(y_test, y_prob)

print("Metricas Gaussian Naive Bayes")
print(f"Accuracy : {acc_g:.4f}")
print(f"Recall   : {rec_g:.4f}")
print(f"F1-Score : {f1_g:.4f}")
print(f"AUC-ROC  : {auc_g:.4f}")

In [ ]:
#Comparamos visualmente las 4 metricas del modelo en un grafico de barras
metricas_g = ['Accuracy', 'Recall', 'F1-Score', 'AUC-ROC']
valores_g  = [acc_g, rec_g, f1_g, auc_g]

plt.figure(figsize=(7, 4))
bars = plt.bar(metricas_g, valores_g, color=['steelblue', 'salmon', 'mediumseagreen', 'mediumpurple'])
plt.ylim(0, 1.1)
plt.title('Metricas - Gaussian Naive Bayes')
plt.ylabel('Valor')
#valor encima de cada barra para facilitar la lectura
for bar, val in zip(bars, valores_g):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.4f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

#Matriz de confusion ---
#La matriz muestra cuantos casos el modelo predijo correctamente e incorrectamente para cada clase
cm_g = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_g, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Maligno', 'Benigno'],
            yticklabels=['Maligno', 'Benigno'])
plt.title('Matriz de Confusion - Gaussian NB')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

## **Bayes - Multinomial**
Para conteos de palabras (ej. clasificacion de texto)

In [ ]:
#Definimos las categorias de noticias a clasificar: articulos de ciencia espacial vs articulos de beisbol
categorias = ['sci.space', 'rec.sport.baseball']

#Descargamos el dataset con remove para limpiar encabezados, pies de pagina
#y citas que no aportan informacion relevante al contenido del articulo
#Esto mejora la calidad del texto antes de vectorizarlo
datos_m = fetch_20newsgroups(
    subset='train',
    categories=categorias,
    remove=('headers', 'footers', 'quotes')  #Eliminamos ruido del texto
)

print(f"Total de documentos: {len(datos_m.data)}")
print(f"Distribucion de clases: {dict(zip(categorias, [sum(datos_m.target==i) for i in range(2)]))}")

#CountVectorizer convierte el texto en matriz de conteo de palabras
#min_df=2: ignoramos palabras que aparecen en menos de 2 documentos (ruido)
#stop_words='english': eliminamos palabras comunes sin valor informativo (the, is, at...)
#max_features=10000: limitamos el vocabulario a las 10000 palabras mas frecuentes

vectores = CountVectorizer(
    min_df=2,
    stop_words='english',
    max_features=10000
)
X_m = vectores.fit_transform(datos_m.data)  #Transformamos el texto a numeros
y_m = datos_m.target                         #Etiquetas: 0 = baseball, 1 = space

print(f"\nForma de la matriz de terminos: {X_m.shape}")
print(f"Vocabulario: {len(vectores.vocabulary_)} palabras")

# Dividimos en entrenamiento y prueba
X_trainm, X_testm, y_trainm, y_testm = train_test_split(
    X_m, y_m, test_size=0.2, random_state=19
)

In [ ]:
#Creamos y entrenamos el modelo Multinomial
modeloM = MultinomialNB()
modeloM.fit(X_trainm, y_trainm)

In [ ]:
#Generamos predicciones usando el modelo Multinomial
y_predim = modeloM.predict(X_testm)
y_probim = modeloM.predict_proba(X_testm)[:, 1]

In [ ]:
#metricas del modelo Multinomial
acc_m = accuracy_score(y_testm, y_predim)
rec_m = recall_score(y_testm, y_predim)
f1_m  = f1_score(y_testm, y_predim)
auc_m = roc_auc_score(y_testm, y_probim)

print("Metricas Multinomial Naive Bayes")
print(f"Accuracy : {acc_m:.4f}")
print(f"Recall   : {rec_m:.4f}")
print(f"F1-Score : {f1_m:.4f}")
print(f"AUC-ROC  : {auc_m:.4f}")

In [ ]:
#Barras de metricas
metricas_m = ['Accuracy', 'Recall', 'F1-Score', 'AUC-ROC']
valores_m  = [acc_m, rec_m, f1_m, auc_m]

plt.figure(figsize=(7, 4))
bars = plt.bar(metricas_m, valores_m, color=['steelblue', 'salmon', 'mediumseagreen', 'mediumpurple'])
plt.ylim(0, 1.1)
plt.title('Metricas - Multinomial Naive Bayes')
plt.ylabel('Valor')
for bar, val in zip(bars, valores_m):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.4f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

#Matriz de confusion
#Muestra cuantos articulos de cada categoria fueron clasificados correctamente
cm_m = confusion_matrix(y_testm, y_predim)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_m, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Baseball', 'Space'],
            yticklabels=['Baseball', 'Space'])
plt.title('Matriz de Confusion - Multinomial NB')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

## **Naive Bayes - Bernoulli**
Para datos binarios/booleanos (presencia o ausencia de caracteristicas)

In [ ]:
#150 muestras de flores con 4 caracteristicas
#(longitud y ancho de sepalo y petalo) y 3 especies posibles
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['target'] = iris.target

# Verificamos si hay valores nulos
print("Valores nulos en Iris:")
print(df_iris.isnull().sum().sum(), "valores nulos en total")

# Verificamos la distribucion de las 3 clases originales
print(f"\nDistribucion original: {dict(zip(iris.target_names, [sum(iris.target==i) for i in range(3)]))}")

#Filtramos solo 2 clases: setosa (0) y versicolor (1)
#Eliminamos virginica (clase 2) para mantener el problema binario
mascara  = iris.target != 2
X_iris   = iris.data[mascara]
y_iris   = iris.target[mascara]

print(f"\nDespues del filtrado: {X_iris.shape[0]} muestras")
print(f"Clases: Setosa={sum(y_iris==0)}, Versicolor={sum(y_iris==1)}")

#Binarizamos las caracteristicas usando la media como umbral
#Si el valor esta por encima de la media -> 1 (caracteristica presente)
#Si el valor esta por debajo de la media -> 0 (caracteristica ausente)
#Esto convierte los valores continuos en datos binarios para BernoulliNB
umbral      = np.mean(X_iris)
binarizador = Binarizer(threshold=umbral)
X_bin       = binarizador.fit_transform(X_iris)

print(f"\nUmbral de binarizacion: {umbral:.4f}")
print(f"Valores unicos despues de binarizar: {np.unique(X_bin)}")

#Dividimos en entrenamiento (80%) y prueba (20%)
X_trainb, X_testb, y_trainb, y_testb = train_test_split(
    X_bin, y_iris, test_size=0.2, random_state=19
)

In [ ]:
#Creamos y entrenamos el modelo Bernoulli
#BernoulliNB calcula la probabilidad de que cada caracteristica binarizada
#sea 1 o 0 para cada especie de flor
modeloB = BernoulliNB()
modeloB.fit(X_trainb, y_trainb)

In [ ]:
#Generamos predicciones y probabilidades del modelo Bernoulli
y_predb = modeloB.predict(X_testb)
y_probb = modeloB.predict_proba(X_testb)[:, 1]  # Probabilidad clase positiva (versicolor)

In [ ]:
#Calculamos todas las metricas del modelo Bernoulli
acc_b = accuracy_score(y_testb, y_predb)
rec_b = recall_score(y_testb, y_predb)
f1_b  = f1_score(y_testb, y_predb)
auc_b = roc_auc_score(y_testb, y_probb)

print("Metricas Bernoulli Naive Bayes (Iris)")
print(f"Accuracy : {acc_b:.4f}")
print(f"Recall   : {rec_b:.4f}")
print(f"F1-Score : {f1_b:.4f}")
print(f"AUC-ROC  : {auc_b:.4f}")

In [ ]:
#Barras de metricas
metricas_b = ['Accuracy', 'Recall', 'F1-Score', 'AUC-ROC']
valores_b  = [acc_b, rec_b, f1_b, auc_b]

plt.figure(figsize=(7, 4))
bars = plt.bar(metricas_b, valores_b, color=['steelblue', 'salmon', 'mediumseagreen', 'mediumpurple'])
plt.ylim(0, 1.1)
plt.title('Metricas - Bernoulli Naive Bayes (Iris)')
plt.ylabel('Valor')
for bar, val in zip(bars, valores_b):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.4f}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

#Matriz de confusion
#Muestra cuantas flores de cada especie fueron clasificadas correctamente
cm_b = confusion_matrix(y_testb, y_predb)
plt.figure(figsize=(5, 4))
sns.heatmap(cm_b, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Setosa', 'Versicolor'],
            yticklabels=['Setosa', 'Versicolor'])
plt.title('Matriz de Confusion - Bernoulli NB (Iris)')
plt.ylabel('Real')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

In [ ]:
#metricas de los 3 modelos juntos para comparar
#su rendimiento de forma visual y clara

modelos  = ['Gaussian\n(breast cancer)', 'Multinomial\n(noticias)', 'Bernoulli\n(iris)']
accuracy = [acc_g, acc_m, acc_b]
recall   = [rec_g, rec_m, rec_b]
f1       = [f1_g,  f1_m,  f1_b]
auc      = [auc_g, auc_m, auc_b]

x     = np.arange(len(modelos))  # Posiciones en el eje X
width = 0.2                       # Ancho de cada barra

fig, ax = plt.subplots(figsize=(10, 5))

#Dibujamos una barra por metrica, desplazada para no solaparse
ax.bar(x - 1.5*width, accuracy, width, label='Accuracy',  color='steelblue')
ax.bar(x - 0.5*width, recall,   width, label='Recall',    color='salmon')
ax.bar(x + 0.5*width, f1,       width, label='F1-Score',  color='mediumseagreen')
ax.bar(x + 1.5*width, auc,      width, label='AUC-ROC',   color='mediumpurple')

ax.set_ylim(0, 1.15)
ax.set_xticks(x)
ax.set_xticklabels(modelos)
ax.set_ylabel('Valor')
ax.set_title('Comparacion de Metricas - Los 3 Modelos Naive Bayes')
ax.legend()
plt.tight_layout()
plt.show()

# Tabla resumen en consola
print("\nRESUMEN COMPARATIVO")
print(f"{'Modelo':<15} {'Accuracy':<12} {'Recall':<12} {'F1-Score':<12} {'AUC-ROC':<12}")
print("-" * 63)
print(f"{'Gaussian':<15} {acc_g:<12.4f} {rec_g:<12.4f} {f1_g:<12.4f} {auc_g:<12.4f}")
print(f"{'Multinomial':<15} {acc_m:<12.4f} {rec_m:<12.4f} {f1_m:<12.4f} {auc_m:<12.4f}")
print(f"{'Bernoulli':<15} {acc_b:<12.4f} {rec_b:<12.4f} {f1_b:<12.4f} {auc_b:<12.4f}")